In [ ]:
# aktiviraj myenv okolje
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

tukaj smo preverili, če je $\tilde{c} = c$

In [ ]:
# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Nastavimo ali bomo praznili ali polnili baterijo
charge = False

In [ ]:
# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 12,           # Osnovna velikost pisave
    'axes.titlesize': 16,      # Velikost naslova grafa
    'axes.labelsize': 14,      # Velikost oznak osi
    'xtick.labelsize': 12,     # Velikost oznak na x osi
    'ytick.labelsize': 12,     # Velikost oznak na y osi
    'legend.fontsize': 12,     # Velikost pisave v legendi
})

In [ ]:
# model, ki ima le mehansko degradacijo
model_DFN_Swell = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        name="Swell",)


models = [model_DFN_Swell]

Parametri

In [ ]:
param = pybamm.ParameterValues("Ai2020") # Pybamm referenca za mehanski model

param.update(
    {"Hydrostatic stress [Pa]": 0,     
     },
    check_already_exists=False ,
    
)

if charge:
    param.set_initial_stoichiometries(0);
# param = pybamm.ParameterValues("OKane2022")

Mreženje

In [ ]:
var_pts = {
    "x_n": 20,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 20,  # positive electrode
    "r_n": 26,  # negative particle
    "r_p": 26,  # positive particle
}

In [ ]:
# Simulacija
if charge:
    experiment = pybamm.Experiment(
        [
            "Charge at 1C until 4.2 V",
            
        ]
    )

else:
    experiment = pybamm.Experiment(
        [
            "Discharge at 1C until 3.0 V",
        ]
    )
    
sim_Swell = pybamm.Simulation(model_DFN_Swell, parameter_values=param, var_pts=var_pts, experiment=experiment)


start = time.time()
sol_Swell = sim_Swell.solve()
end = time.time()
print(f"Simulacija Swell zaključena v {end - start:.2f} s.")



## Napetosti

$$
\sigma_{rr}^c(r) = \frac{2 \Omega E_p}{3(1 - \nu_p)} \left( \frac{1}{r_p^3} \int_0^{r_p} \tilde{c} r^2 \, dr - \frac{1}{r^3} \int_0^r \tilde{c} r^2 \, dr \right),
$$

$$
\sigma_{\theta\theta}^c(r) = \frac{\Omega E_p}{3(1 - \nu_p)} \left( \frac{2}{r_p^3} \int_0^{r_p} \tilde{c} r^2 \, dr + \frac{1}{r^3} \int_0^r \tilde{c} r^2 \, dr - \tilde{c} \right),
$$

$$
\sigma_h^c(r) = \frac{\sigma_{rr}^c + 2\sigma_{\theta\theta}^c}{3} = \frac{2 \Omega E_p}{3(1 - \nu_p)} \left( \frac{1}{r_p^3} \int_0^{r_p} \tilde{c} r^2 \, dr - \frac{\tilde{c}}{3} \right)
$$

$$\tilde{c} = c_s - c_{s0}$$

$r_p$ ... radij delca \
$𝐸_𝑝$ ... Youngov modul delca \
$𝑣_𝑝$ ... Poissonov količnik delca \
$\sigma_{rr}^c$ ... radialna napetost delca \
$\sigma_{\theta\theta}^c$ ... tangencialna napetost delca \
$c_s$ ... koncentracija v trdnini \
$c_{s0}$ ... koncentracija v začetnem oz. neobremenjenem stanju 

PyBaMM:

$$
\sigma_h^c(r) = \frac{2 \Omega E_p}{3(1 - \nu_p)} \left( \frac{1}{r_p^3} \int_0^{r_p} c r^2 \, dr - \frac{c}{3} \right)
$$

$$\boxed{\tilde{c} = c}$$


### Hidrostatična

#### Delec

In [ ]:
#____Negativen delec____
E_p = param["Negative electrode Young's modulus [Pa]"]
nu_p = param["Negative electrode Poisson's ratio"]
Omega = param["Negative electrode partial molar volume [m3.mol-1]"]

c_s0 = param["Initial concentration in negative electrode [mol.m-3]"]
c_s = sol_Swell["Negative particle concentration [mol.m-3]"].data
c_tilde = c_s - c_s0
c_s_rav = sol_Swell["R-averaged negative particle concentration [mol.m-3]"].data

r_p = param["Negative particle radius [m]"]
r = pybamm.SpatialVariable("r", domain="particle", coord_sys="spherical polar")

sigma_h_n = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav - c_s0)/3 - c_tilde/3)
sigma_h2 = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav)/3 - c_s/3) # PyBaMM

In [ ]:
print(f"c_s: {c_s.shape}")
print(f"c_tilde: {c_tilde.shape}")
print(f"c_s_rav: {c_s_rav.shape}")
print(f"sigma_h: {sigma_h_n.shape}")
print(f"sigma_h2: {sigma_h2.shape}")

**Primerjava modelov**

##### Anodni delec

In [ ]:
# Seznami parametrov in rešitev za vse tri modele
param_list = [param]
sol_list = [sol_Swell]
sigma_h_neg_list = []

# Zanka čez vse modele
for param_i, sol_i in zip(param_list, sol_list):
    # Parametri modela
    E_p = param_i["Negative electrode Young's modulus [Pa]"]
    nu_p = param_i["Negative electrode Poisson's ratio"]
    Omega = param_i["Negative electrode partial molar volume [m3.mol-1]"]
    c_s0 = param_i["Initial concentration in negative electrode [mol.m-3]"]
    
    # Podatki iz rešitve
    c_s = sol_i["Negative particle concentration [mol.m-3]"].data
    c_tilde = c_s - c_s0
    c_s_rav = sol_i["R-averaged negative particle concentration [mol.m-3]"].data
    
    
    # Izračun sigma_h
    sigma_h_n = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav - c_s0)/3 - c_tilde/3)
    sigma_h_neg_list.append(sigma_h_n)
    sigma_h2 = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav)/3 - c_s/3) # PyBaMM
    sigma_h_neg_list.append(sigma_h2)

# Poimenuj rezultate, če želiš individualni dostop
sigma_h_n_Swell, sigma_h_n_Swell2 = sigma_h_neg_list


In [ ]:
print(f"sigma_h_Swell: {sigma_h_n_Swell.shape}")
print(f"sigma_h_Clerici: {sigma_h_n_Swell2.shape}")

In [ ]:
# Poišči najmanjše število frameov med vsemi tremi modeli
min_frames = min(sigma_h_n_Swell.shape[2], sigma_h_n_Swell2.shape[2])

# Odreži vse na enako dolžino (min_frames)
sigma_h_Swell_cut = sigma_h_n_Swell[:, :, :min_frames]
sigma_h_Swell2_cut = sigma_h_n_Swell2[:, :, :min_frames]


# Uporabi za primerjavo
y_val_n = [sigma_h_Swell_cut, sigma_h_Swell2_cut]

# Izreži podatke za prostorski indeks 0 (ali poljubni presek)
sigma_h_n_swell = sigma_h_Swell_cut[:, 0, :]     # shape: (26, 259)
sigma_h_n_clerici = sigma_h_Swell2_cut[:, 0, :] # shape: (26, 259)



In [ ]:
# --- Plot ---
r_max = param["Negative particle radius [m]"]
x = np.linspace(0, r_max, 26)

# --- Priprava podatkov ---
model_labels = ["Swell", "Swell2"]
line_styles = ["-", "--"]
colors = ["C0", "C1"]
sigma_vals = [s[:, 0, :] for s in y_val_n]  # izvlečemo časovni presek pri x=0
time_array = sol_Swell["Time [s]"].entries[:sigma_vals[0].shape[1]]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Radial position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"Negative particle $\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
if save_ani:
    if charge:
        ani.save("slike/Sigma_h_negative_particle_charge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_negative_particle_charge.gif'.")
    else:
        ani.save("slike/Sigma_h_negative_particle_discharge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_negative_particle_discharge.gif'.")


##### Katodni delec

In [ ]:
# Seznami parametrov in rešitev za vse tri modele
param_list = [param]
sol_list = [sol_Swell]
sigma_h_pos_list = []

# Zanka čez vse modele
for param_i, sol_i in zip(param_list, sol_list):
    # Parametri modela
    E_p = param_i["Positive electrode Young's modulus [Pa]"]
    nu_p = param_i["Positive electrode Poisson's ratio"]
    Omega = param_i["Positive electrode partial molar volume [m3.mol-1]"]
    c_s0 = param_i["Initial concentration in positive electrode [mol.m-3]"]
    
    # Podatki iz rešitve
    c_s = sol_i["Positive particle concentration [mol.m-3]"].data
    c_tilde = c_s - c_s0
    c_s_rav = sol_i["R-averaged negative particle concentration [mol.m-3]"].data
    
    
    # Izračun sigma_h
    sigma_h_p = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav - c_s0)/3 - c_tilde/3)
    sigma_h_pos_list.append(sigma_h_p)
    sigma_h2 = 2 * Omega * E_p / (3 * (1 - nu_p)) * ((c_s_rav)/3 - c_s/3) # PyBaMM
    sigma_h_pos_list.append(sigma_h2)

# Poimenuj rezultate, če želiš individualni dostop
sigma_h_p_Swell, sigma_h_p_Swell2 = sigma_h_pos_list


In [ ]:
print(f"sigma_h_Swell: {sigma_h_p_Swell.shape}")
print(f"sigma_h_Clerici: {sigma_h_p_Swell2.shape}")

In [ ]:
# Poišči najmanjše število frameov med vsemi tremi modeli
min_frames = min(sigma_h_p_Swell.shape[2], sigma_h_p_Swell2.shape[2])

# Odreži vse na enako dolžino (min_frames)
sigma_h_Swell_cut = sigma_h_p_Swell[:, :, :min_frames]
sigma_h_Swell2_cut = sigma_h_p_Swell2[:, :, :min_frames]



# Uporabi za primerjavo
y_val_p = [sigma_h_Swell_cut, sigma_h_Swell2_cut]

# Izreži podatke za prostorski indeks 0 (ali poljubni presek)
sigma_h_p_swell = sigma_h_Swell_cut[:, 0, :]     # shape: (26, 259)
sigma_h_p_clerici = sigma_h_Swell2_cut[:, 0, :] # shape: (26, 259)



In [ ]:
# --- Plot ---
r_max = param["Positive particle radius [m]"]
x = np.linspace(0, r_max, 26)

# --- Priprava podatkov ---
model_labels = ["Swell", "Swell2"]
line_styles = ["-", "--"]
colors = ["C0", "C1"]
sigma_vals = [s[:, 0, :] for s in y_val_p]  # izvlečemo časovni presek pri x=0
time_array = sol_Swell["Time [s]"].entries[:sigma_vals[0].shape[1]]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Radial position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"Positive particle $\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
if save_ani:
    if charge:
        ani.save("slike/Sigma_h_positive_particle_charge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_positive_particle_charge.gif'.")
    else:
        ani.save("slike/Sigma_h_positive_particle_discharge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_positive_particle_discharge.gif'.")

#### Elektroda

##### Anoda

In [ ]:
# Poišči najmanjše število frameov med vsemi tremi modeli
min_frames = min(sigma_h_n_Swell.shape[2], sigma_h_n_Swell2.shape[2])

# Odreži vse na enako dolžino (min_frames)
sigma_h_Swell_cut = sigma_h_n_Swell[:, :, :min_frames]
sigma_h_Swell2_cut = sigma_h_n_Swell2[:, :, :min_frames]

# Uporabi za primerjavo
y_val_n = [sigma_h_Swell_cut, sigma_h_Swell2_cut]


In [ ]:
# --- Plot ---
x_max = param["Negative electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["Swell", "Swell2"]
line_styles = ["-", "--"]
colors = ["C0", "C1"]
sigma_vals = [s[-1, :, :] for s in y_val_n]  # gledamo surface napetost
time_array = sol_Swell["Time [s]"].entries[:sigma_vals[0].shape[1]]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Negative electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
if save_ani:
    if charge:
        ani.save("slike/Sigma_h_negative_electrode_charge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_negative_electrode_charge.gif'.")
    else:
        ani.save("slike/Sigma_h_negative_electrode_discharge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_negative_electrode_discharge.gif'.")

##### Katoda

In [ ]:
# Poišči najmanjše število frameov med vsemi tremi modeli
min_frames = min(sigma_h_p_Swell.shape[2], sigma_h_p_Swell2.shape[2])

# Odreži vse na enako dolžino (min_frames)
sigma_h_Swell_cut = sigma_h_p_Swell[:, :, :min_frames]
sigma_h_Swell2_cut = sigma_h_p_Swell2[:, :, :min_frames]



# Uporabi za primerjavo
y_val_p = [sigma_h_Swell_cut, sigma_h_Swell2_cut]

# Izreži podatke za površinsko napetost
sigma_h_p_swell = sigma_h_Swell_cut[-1, :, :]     # shape: (20, 259)
sigma_h_p_clerici = sigma_h_Swell2_cut[-1, :, :] # shape: (20, 259)



In [ ]:
# --- Plot ---
x_max = param["Positive electrode thickness [m]"]
x = np.linspace(0, x_max, var_pts['x_n'])  # 20 points for electrode thickness

# --- Priprava podatkov ---
model_labels = ["Swell", "Swell2"]
line_styles = ["-", "--"]
colors = ["C0", "C1"]
sigma_vals = [s[-1, :, :] for s in y_val_p]  # izvlečemo časovni presek pri x=0
time_array = sol_Swell["Time [s]"].entries[:sigma_vals[0].shape[1]]  # krajši časovni vektor

# --- Priprava figure ---
fig, ax = plt.subplots(figsize=(8, 5))
lines = [ax.plot([], [], label=label, linestyle=ls, color=col, linewidth=2)[0]
         for label, ls, col in zip(model_labels, line_styles, colors)]

# Nastavi meje
x_margin = 0.05 * (x[-1] - x[0])
ax.set_xlim(x[0] - x_margin, x[-1] + x_margin)

y_all = np.stack(sigma_vals)
y_min, y_max = np.min(y_all), np.max(y_all)
y_margin = 0.1 * (y_max - y_min)
ax.set_ylim(y_min - y_margin, y_max + y_margin)

ax.set_xlabel("Positive electrode position [m]")
ax.set_ylabel(r"$\sigma_h$ [Pa]")
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9)

# --- Inicializacija ---
def init():
    for line in lines:
        line.set_data([], [])
    ax.set_title("Starting...")
    return lines

# --- Posodabljanje ---
def update(frame):
    for line, sigma in zip(lines, sigma_vals):
        line.set_data(x, sigma[:, frame])
    ax.set_title(f"$\sigma_h$ at t={time_array[frame]:.2f} s")
    return lines

# --- Animacija ---
ani = FuncAnimation(fig, update, frames=sigma_vals[0].shape[1], init_func=init, blit=True)
plt.close()

# Za prikaz v Jupyterju:
HTML(ani.to_jshtml())


In [ ]:
if save_ani:
    if charge:
        ani.save("slike/Sigma_h_positive_electrode_charge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_positive_electrode_charge.gif'.")
    else:
        ani.save("slike/Sigma_h_positive_electrode_discharge.gif", writer='pillow', fps=10)
        print("Saved animation as 'Sigma_h_positive_electrode_discharge.gif'.")